In [ ]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = 'ComfyUI'
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    
    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -U --pre comfyui-manager

In [ ]:
#@title Download some models/checkpoints/vae or custom comfyui nodes
# ==========================================
# Download all required models
# ==========================================
import os
import subprocess
from pathlib import Path

def get_hf_token():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HUGGINGFACE_TOKEN')
    except Exception:
        pass
    return token

hf_token = get_hf_token()

def download(url, dest, auth=False):
    Path(dest).parent.mkdir(parents=True, exist_ok=True)
    cmd = ['wget', '-c', url, '-O', dest]
    if auth:
        if hf_token:
            cmd = ['wget', '-c', '--header', f'Authorization: Bearer {hf_token}', url, '-O', dest]
        else:
            print('Warning: auth requested but no Hugging Face token found, trying without auth.')
    print('Downloading', dest)
    subprocess.run(cmd, check=True)

# UNET model (flux1-dev-fp8)
unet_url = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors'
unet_path = '/content/ComfyUI/models/unet/flux1-dev-fp8.safetensors'
download(unet_url, unet_path, auth=True)

# VAE
vae_url = 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors'
vae_path = '/content/ComfyUI/models/vae/ae.safetensors'
download(vae_url, vae_path, auth=True)

# Dual CLIP
download('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors', '/content/ComfyUI/models/clip/clip_l.safetensors', auth=False)
download('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors', '/content/ComfyUI/models/clip/t5xxl_fp8_e4m3fn.safetensors', auth=False)

# PuLID Flux
download('https://huggingface.co/guozinan/PuLID/resolve/main/pulid_flux_v0.9.1.safetensors', '/content/ComfyUI/models/pulid/pulid_flux_v0.9.1.safetensors', auth=True)

# EVA-CLIP
download('https://huggingface.co/QuanSun/EVA-CLIP/resolve/main/EVA02_CLIP_L_336_psz14_s6B.pt', '/content/ComfyUI/models/clip_vision/EVA02_CLIP_L_336_psz14_s6B.pt', auth=True)

# InsightFace - AntelopeV2 (tải zip chuẩn tên file cho ComfyUI)
insightface_dir = '/content/ComfyUI/models/insightface/models/antelopev2'
Path(insightface_dir).mkdir(parents=True, exist_ok=True)
download('https://huggingface.co/MonsterMMORPG/tools/resolve/main/antelopev2.zip', '/tmp/antelopev2.zip', auth=False)
subprocess.run(['unzip', '-o', '/tmp/antelopev2.zip', '-d', insightface_dir], check=True)

# CLIPSeg
download('https://huggingface.co/CIDAS/clipseg-rd64-refined/resolve/main/pytorch_model.bin', '/content/ComfyUI/models/clipseg/pytorch_model.bin', auth=False)

# Ultralytics face bbox
download('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt', '/content/ComfyUI/models/ultralytics/bbox/face_yolov8m.pt', auth=False)

print('--- ĐÃ TẢI XONG TOÀN BỘ MODEL CHO WORKFLOW CỦA BẠN! ---')


In [ ]:
#@title Install custom nodes for ComfyUI
# Di chuyển vào thư mục custom_nodes của ComfyUI
%cd /content/ComfyUI/custom_nodes/

# Cài đặt bộ công cụ Impact Pack (Chứa FaceDetailer)
!git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git
%cd ComfyUI-Impact-Pack
!pip install -r requirements.txt
%cd ..

# Cài đặt Impact Subpack (companion cho Impact Pack)
!git clone https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git
%cd ComfyUI-Impact-Subpack
!pip install -r requirements.txt
%cd ..

# Cài đặt bộ PuLID Flux Enhanced
!git clone https://github.com/sipie800/ComfyUI-PuLID-Flux-Enhanced.git
%cd ComfyUI-PuLID-Flux-Enhanced
!pip install -r requirements.txt
%cd ..

# Cài đặt WAS Node Suite (Chứa node CLIPSEG2 trong workflow của ông)
!git clone https://github.com/WASasquatch/was-node-suite-comfyui.git

# Quay trở lại thư mục chính của ComfyUI
%cd /content/ComfyUI


### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --enable-manager --dont-print-server